# 长期财政原值账本验收
原表内部数值核查与跨期可比性分开。本笔记不运行回归、不插值、不绘图。完整核查见verification_checks.csv与verification_receipt.json；原表读取逻辑见项目scripts/build_longitudinal_ledger.py。

In [1]:
from pathlib import Path
import csv, json, hashlib
base = Path.cwd()
if not (base / 'raw_value_ledger.csv').exists():
    base = base / 'research/data/长期财政数据建设_20261010'
def read(name):
    with (base / name).open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))
ledger = read('raw_value_ledger.csv')
sources = read('source_registry.csv')
students = read('student_candidate_series.csv')
assert len({r['record_id'] for r in ledger}) == len(ledger)
assert not ({r['source_id'] for r in ledger} - {r['source_id'] for r in sources})
assert sum(r['province'] != '全国' for r in students) == 1856
assert all(not r['accepted_value'] for r in ledger if not r['numeric_candidate'])
assert all(r['model_eligible'] == 'False' for r in ledger)
print(json.dumps(json.loads((base / 'verification_receipt.json').read_text(encoding='utf8')), ensure_ascii=False, indent=2))


{
  "checks": {
    "source_ids_unique": true,
    "record_ids_unique": true,
    "source_foreign_keys_complete": true,
    "student_province_grain_unique": true,
    "student_expected_1856_province_rows": true,
    "panel_grain_unique": true,
    "overall_CPI_only": true,
    "missing_not_imputed": true,
    "no_model_admission_claim": true,
    "protected_hashes": true,
    "2012_volume_continuation_headers_distinct": true,
    "2012_volume_continuation_original_rows": true
  },
  "ledger_rows": 239276,
  "numeric_coverage_rows": 460,
  "student_invalid_text_cells": 8,
  "student_blank_grade_cells": 169,
  "student_failed_or_uncheckable_checks": 179,
  "unresolved_finance_region_rows": 16,
  "finance_original_component_checks": 4693,
  "finance_original_component_status": {
    "uncheckable_invalid_or_missing": 60,
    "pass_rounding": 4633
  }
}
